# Assignment 2 - Research Forecasting Extensions

This notebook extends the Assignment 2 study into a comparative forecasting research framework. It explores classical interpretable models, LightGBM, probabilistic quantile forecasts, lightweight sequence models, cluster-aware forecasting, and graph/spatial future-work ideas while preserving the leakage-safe pipeline.

## Research Framing

The goal is exploratory, not leaderboard optimization. The study compares inductive biases: linear regularized models, tree ensembles, boosting, hierarchical allocation, cluster-conditioned models, quantile uncertainty, and small sequence models. The comparison emphasizes accuracy, stability, interpretability, scalability, granularity, and operational realism.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "data").exists() and (candidate / "solution" / "assignment_2" / "src").exists():
        repo_root = candidate
        break
else:
    raise FileNotFoundError("Could not find ECESIS repository root from notebook path.")

src_dir = repo_root / "solution" / "assignment_2" / "src"
outputs_dir = repo_root / "solution" / "assignment_2" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(src_dir))

pd.set_option("display.max_columns", None)


## Research Experiment Artifacts

The experiment runner uses the bounded prototype data size. To keep notebook execution stable, this notebook reads generated artifacts by default. Set `RUN_RESEARCH = True` in the next cell to rerun the classical, quantile, and sequence experiments.

In [ ]:
RUN_RESEARCH = False

if RUN_RESEARCH:
    from research_experiments import run_research_experiments
    research = run_research_experiments(outputs_dir, n_buses=20)
    result_shapes = {k: v.shape for k, v in research.items()}
else:
    result_shapes = {
        p.name: pd.read_csv(p).shape
        for p in sorted(outputs_dir.glob("research_*.csv"))
    }

result_shapes

## Classical and Boosting Models

Ridge and ElasticNet provide interpretable linear baselines and coefficient inspection. RandomForest and LightGBM provide nonlinear alternatives. Comparing them with existing XGBoost, CatBoost, HGB, and hierarchical outputs shows how much nonlinear structured modeling matters.

In [ ]:
classical_log = pd.read_csv(outputs_dir / "research_classical_model_log.csv")
classical_eval = pd.read_csv(outputs_dir / "research_classical_evaluation_summary.csv")
classical_log, classical_eval.sort_values(["horizon", "level", "wmape"]).head(20)

## Quantile Forecasting and Uncertainty

LightGBM quantile models produce P10/P50/P90 forecasts. The interval width is an operational uncertainty signal: wider bands around peak demand indicate less certain load outcomes and can support reserve or risk-aware planning.

In [ ]:
quantile_summary = pd.read_csv(outputs_dir / "research_quantile_summary.csv")
quantile_log = pd.read_csv(outputs_dir / "research_quantile_model_log.csv")
quantile_summary, quantile_log

## Lightweight Sequence Models

TCN, GRU, and LSTM models are included as small exploratory architectures. They use sliding windows and temporal ordering, but are not tuned. In this prototype they are best interpreted as evidence about compute and stability tradeoffs rather than final candidates.

In [ ]:
sequence_log = pd.read_csv(outputs_dir / "research_sequence_model_log.csv")
sequence_log

## Comprehensive Comparison

This table combines the previous advanced results with the added classical and sequence models. It includes next-day, next-week, and next-month where available.

In [ ]:
comparison = pd.read_csv(outputs_dir / "research_comprehensive_model_comparison.csv")
comparison.sort_values(["horizon", "level", "wmape"]).head(40)

## Spatial and Graph-Aware Future Work

A graph-aware extension would connect buses to zones and possibly zones to neighboring zones. Without topology data, the current implemented proxy is hierarchical bus-zone modeling plus cluster-aware structure. Future graph features could include neighboring-zone lagged load, zone residual smoothing, bus-zone hierarchy embeddings, or graph neural networks once physical connectivity is available.

In [ ]:
future_work = pd.read_csv(outputs_dir / "research_future_work_notes.csv")
future_work

## Research-Style Takeaways

The forecasting problem has strong cyclical temporal structure and hierarchical spatial organization. Boosting models are effective because lag, rolling, cyclical, zone, bus, and cluster features interact nonlinearly. Hierarchical forecasting can stabilize noisy bus-level behavior. Clustering acts as structural regularization by grouping similar operational demand archetypes. Deep temporal models may capture richer sequences, but they require more engineering and compute. Different model families offer different tradeoffs among predictive power, interpretability, uncertainty estimation, scalability, and operational practicality.

In [ ]:
sorted(p.name for p in outputs_dir.glob("research_*.csv"))